# N105 · 字符串分割、解码与模式匹配

**目标：** 把前缀能否完成和模式状态作为DP状态。

**先修：** N098, N066, N104。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 词典分割；多位解码；0的合法性；通配符与正则语义差异。

## 从问题到算法

分词是前缀可达性，解码是前缀路径计数，通配符与正则是模式状态自动机。两种星号语义不同：通配符*匹配任意字符串；简化正则x*重复前一个元素。匹配必须覆盖整个字符串，不能把搜索某个子串当成完整匹配。

## 最小实现

**本章接口：** `word_break(s, words)`、`num_decodings(s)`、`wildcard_match(s, p)`、`regex_match(s, p)`

In [1]:
def word_break(s,words):
    words=set(words); words.discard(''); dp=[True]+[False]*len(s)
    lengths={len(w) for w in words}
    for i in range(1,len(s)+1): dp[i]=any(i>=k and dp[i-k] and s[i-k:i] in words for k in lengths)
    return dp[-1]

def num_decodings(s):
    if not s: return 0
    dp=[0]*(len(s)+1); dp[0]=1
    for i in range(1,len(s)+1):
        if s[i-1]!='0': dp[i]+=dp[i-1]
        if i>=2 and '10'<=s[i-2:i]<='26': dp[i]+=dp[i-2]
    return dp[-1]

def wildcard_match(s,p):
    dp=[False]*(len(p)+1); dp[0]=True
    for j,c in enumerate(p,1): dp[j]=dp[j-1] and c=='*'
    for x in s:
        old=dp; dp=[False]*(len(p)+1)
        for j,c in enumerate(p,1): dp[j]=(dp[j-1] or old[j]) if c=='*' else old[j-1] and (c=='?' or c==x)
    return dp[-1]

def regex_match(s,p):
    if p.startswith('*') or '**' in p: raise ValueError('star requires a preceding atom')
    dp=[[False]*(len(p)+1) for _ in range(len(s)+1)]; dp[0][0]=True
    for i in range(len(s)+1):
        for j in range(1,len(p)+1):
            if p[j-1]=='*':
                dp[i][j]=dp[i][j-2] or (i>0 and (p[j-2]=='.' or p[j-2]==s[i-1]) and dp[i-1][j])
            elif i>0: dp[i][j]=(p[j-1]=='.' or p[j-1]==s[i-1]) and dp[i-1][j-1]
    return dp[-1][-1]

## 正确性、前提与复杂度

分词最后一段必须属于字典；解码最后一码长度只可能1或2，且0不可独立解码。通配符星号分为匹配空串或消耗一个字符；正则星号分为零次前一元素或再消耗一次，二者下标移动不同。

**代价：** 匹配DP O(|s||p|)时间；通配符O(|p|)辅助，正则此处O(|s||p|)辅助。分词含Python子串复制与哈希，保守O(n∑候选长度)字符工作量。解码输入为十进制数字串；简化正则只含普通字符、点和星号。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

s='226'; rows=[]; dp=[1]
for i in range(1,len(s)+1):
    one=dp[i-1] if s[i-1]!='0' else 0; two=dp[i-2] if i>=2 and '10'<=s[i-2:i]<='26' else 0
    dp.append(one+two); rows.append((s[:i],one,two,dp[i]))
show_table(['前缀','一位末码','两位末码','总数'],rows)

前缀,一位末码,两位末码,总数
2,1,0,1
22,1,1,2
226,2,1,3


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert word_break('leetcode',['leet','code']) and not word_break('catsandog',['cats','dog','sand','and','cat'])
assert num_decodings('06')==0 and num_decodings('226')==3
assert regex_match('aab','c*a*b') and not regex_match('mississippi','mis*is*p*.')
assert regex_match('','a*') and wildcard_match('','**')
from itertools import product
from functools import cache
import re
@cache
def wildcard_ref(s,p):
    if not p: return not s
    if p[0]=='*': return wildcard_ref(s,p[1:]) or bool(s) and wildcard_ref(s[1:],p)
    return bool(s) and p[0] in ('?',s[0]) and wildcard_ref(s[1:],p[1:])
for n in range(4):
    for a in product('ab',repeat=n):
        s=''.join(a)
        for p in product('ab?*',repeat=3):
            p=''.join(p); assert wildcard_match(s,p)==wildcard_ref(s,p)
        for atoms in product(('a','b','.','a*','b*','.*'),repeat=2):
            p=''.join(atoms); assert regex_match(s,p)==bool(re.fullmatch(p,s))
print('N105: 所有本章断言通过')

N105: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 比较*在通配符和简化正则中的不同含义。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 给每类匹配写独立契约。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 139. Word Break（canonical）
- 91. Decode Ways（canonical）
- 44. Wildcard Matching（canonical）
- 10. Regular Expression Matching（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。